# Lesson 20: Web Fundamentals for Non-Developers

**Time:** ~30 minutes | **Cost:** $0 (no API calls)

In the next lesson, you'll see the product's web interface — a React frontend talking to a FastAPI backend. This lesson teaches the concepts you need to understand what's happening.

You don't need to become a web developer. You need to understand the **architecture** so you can direct Claude Code to make changes.

## What is a Web App?

A web app has two parts:
- **Frontend** (client) — what you see in the browser. HTML, CSS, JavaScript. In our project: React.
- **Backend** (server) — the engine that processes requests. Python code. In our project: FastAPI + our agents.

```
[Your Browser]  ↔  [Backend Server]
   Frontend           Python + Agents
   (React)            (FastAPI)
```

Analogy: A restaurant.
- **Frontend** = the dining room (menu, tables, waiters)
- **Backend** = the kitchen (chefs, ingredients, recipes)
- The waiter carries your **request** to the kitchen and brings back the **response**

When you type a message in the chat interface and click send, you're the customer ordering. The message goes to the backend (kitchen), the agent team processes it (chefs cook), and the response comes back to your screen (waiter delivers the food).

## What is an API?

**API** (Application Programming Interface) — a set of **endpoints** (URLs) that the backend exposes for the frontend to call.

Example from our project:

| Endpoint | Method | What it does |
|----------|--------|-------------|
| `/api/articles` | GET | List all articles |
| `/api/articles/{id}` | POST | Update an article |
| `/teams/seo-workspace/runs` | POST | Send a chat message to the team |
| `/health` | GET | Check if server is running |

**GET** = "give me data" (like reading a menu)

**POST** = "here's data, process it" (like placing an order)

Every API call is a **request** (from frontend) and a **response** (from backend):

```
Frontend: "GET /api/articles"
Backend:  [reads content/articles.json, returns list]
Frontend: displays the article list in the sidebar
```

You already understand this pattern — it's what the DataForSEO tool does! When the Content Writer searches the web, it makes API calls to DataForSEO's endpoints. Same concept, different server.

In [ ]:
# API calls are just structured requests
# Here's what happens when the frontend asks for articles

# The frontend sends this:
request = {
    "method": "GET",
    "url": "/api/articles",
    "headers": {"Content-Type": "application/json"},
}

# The backend processes it and returns this:
response = {
    "status": 200,  # 200 = OK
    "body": [
        {"id": "on-page-seo-meta-tags", "topic": "On-Page SEO Meta Tags", "status": "review"},
        {"id": "internal-linking", "topic": "Internal Linking Strategy", "status": "review"},
    ]
}

print(f"Request: {request['method']} {request['url']}")
print(f"Response status: {response['status']}")
print(f"Articles returned: {len(response['body'])}")
for article in response['body']:
    print(f"  - {article['topic']} ({article['status']})")

## JSON — The Language of APIs

Frontend and backend communicate in **JSON** — the same format you've been using with `json.loads()` and `json.dumps()` since Lesson 12.

```json
{
    "topic": "SEO Tips",
    "word_count": 1500,
    "status": "review",
    "keywords": ["seo", "tips", "beginners"]
}
```

This is why all our storage functions return JSON strings — they're designed to work with both agents AND web APIs.

## How Frontend and Backend Connect

Our project runs two servers:
- **Backend**: `python output/backend/serve.py` → runs on **port 7777**
- **Frontend**: `cd output/frontend && npm run dev` → runs on **port 5173**

The frontend (port 5173) needs to talk to the backend (port 7777). The Vite dev server (frontend's build tool) **proxies** (forwards) API requests:

```
Browser (port 5173) → Vite proxy → Backend (port 7777)
                        ↓
              Forwards /api/*, /teams/*, /health
```

This means:
- You open `http://localhost:5173` in your browser
- When React sends a request to `/api/articles`, Vite forwards it to `http://localhost:7777/api/articles`
- The response comes back through the same path

**You don't need to configure this** — it's already set up in `output/frontend/vite.config.js`. But understanding it helps when debugging ("why can't the frontend reach the backend?").

## SSE — How Streaming Works

When you chat with ChatGPT, text appears word by word. That's **SSE** (Server-Sent Events) — a way for the server to **stream** data to the browser in real time.

Our chat interface uses the same technique:
1. Frontend sends your message via POST
2. Backend starts the agent team
3. As the agent generates text, the backend sends it **chunk by chunk** via SSE
4. Frontend displays each chunk as it arrives → you see text appearing in real time

Without SSE: you'd wait 30–90 seconds staring at a spinner, then get the full response at once.

With SSE: text starts appearing within seconds, streaming as the agent writes.

```
Frontend: "Write an article about SEO tips" (POST)
Backend:  SSE chunk: "# SEO Tips..."
Backend:  SSE chunk: "\n\nSearch engine optimization..."
Backend:  SSE chunk: " is essential for..."
          ... (continues streaming)
Backend:  SSE chunk: [DONE]
```

## Lesson 20 Summary

| Concept | What it means | In our project |
|---------|--------------|----------------|
| **Frontend** | Browser-side UI (React) | `output/frontend/` |
| **Backend** | Server-side logic (Python) | `output/backend/serve.py` |
| **API** | Endpoints the frontend calls | `/api/articles`, `/teams/...` |
| **GET / POST** | Read data / Send data | List articles / Send chat message |
| **JSON** | Data format for communication | Same format from storage tools |
| **Proxy** | Vite forwards requests to backend | Port 5173 → port 7777 |
| **SSE** | Real-time streaming | Chat text appears word by word |

**What you DON'T need to know**: How React renders components, how FastAPI handles routing internally, how SSE is implemented in JavaScript.

**What you DO need to know**: The architecture — what talks to what, and where data flows. This is enough to direct Claude Code.

**Next lesson:** The Web Interface — seeing all of this in action with our actual product.

## Exercise

Match each component to its role:

In [ ]:
# Exercise: Match each component to its role

components = {
    "React": "___",           # frontend or backend?
    "FastAPI": "___",         # frontend or backend?
    "serve.py": "___",        # what does it do?
    "/api/articles": "___",   # what kind of thing is this?
    "port 7777": "___",       # what runs here?
    "port 5173": "___",       # what runs here?
    "SSE": "___",             # what does it enable?
    "JSON": "___",            # what is it used for?
}

for component, role in components.items():
    print(f"  {component}: {role}")

<details>
<summary>Click to reveal answers</summary>

- **React**: frontend (browser UI)
- **FastAPI**: backend (Python web framework)
- **serve.py**: backend entry point (starts the web server with agents)
- **/api/articles**: API endpoint (URL the frontend calls to get article data)
- **port 7777**: backend server
- **port 5173**: frontend dev server
- **SSE**: real-time streaming (text appears word by word)
- **JSON**: data format for API communication

</details>